## Imports

In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm.auto import tqdm
import os
import sys
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize   
import seaborn as sns

/home/le/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [ ]:
# --- File Paths ---
# Updated for v4: Using cleaned sentence-level entity-linked data (163,139 sentences from 19,659 articles)
TRAIN_DATA_PATH = 'data/01_preprocessed/sentences_entity_linked_cleaned.parquet'
# Updated for v4: Using final dictionary with 1,109 keywords (7 categories)
KEYWORD_DICTIONARY_PATH = 'results/phase i/13_final_dictionary_v4.csv'
OUTPUT_PATH = 'results/phase ii/21_provisionally_labeled_train_set_v4.parquet'

# --- Embedding Cache Paths ---
# NOTE: Delete these files if they exist from previous runs with different datasets
SENTENCE_EMBEDDINGS_PATH = 'embeddings/sentence_embeddings_v4.pt'

KEYWORD_EMBEDDINGS_PATH = 'embeddings/keyword_embeddings_v4.pt'

In [ ]:
# Display sample of sentence-level data
sentence_sample = pd.read_parquet(TRAIN_DATA_PATH)
print(f"Sample of sentence-level dataset:")
print(f"Total sentences: {len(sentence_sample):,}")
print(f"Unique articles: {sentence_sample['article_id'].nunique():,}")
sentence_sample.head()

,title,summary,section,keywords,published_date,url,article_id,title_length,summary_length,total_length,word_count
25,Council Plans Session on Deutsche Bank Fire,The City Council plans an extensive hearing ne...,New York,"['Deutsche Bank Building (NYC)', 'World Trade ...",2008-01-02 05:00:00+00:00,https://www.nytimes.com/2008/01/02/nyregion/02...,99458,43,164,207,29
111,Manhattan: Holocaust Case Settled,A federal judge on Monday approved a settlemen...,New York,"['Nazi Policies Toward Jews and Minorities', '...",2008-01-08 05:00:00+00:00,https://www.nytimes.com/2008/01/08/nyregion/08...,99570,33,624,657,96
130,Demolition to Resume at Deutsche Bank Site,Work was halted in August after two firefighte...,New York,"['Deutsche Bank Building (NYC)', 'World Trade ...",2008-01-09 05:00:00+00:00,https://www.nytimes.com/2008/01/09/nyregion/09...,99589,42,97,139,16
234,Fire Department Alters Its Inspection Schedule,The move is part of a series of changes made i...,New York,"['Deutsche Bank Building (NYC)', 'Fires and Fi...",2008-01-14 05:00:00+00:00,https://www.nytimes.com/2008/01/14/nyregion/14...,99724,46,163,209,30
249,Open a New Window: A Tower With a View,A plan for Verizon’s tower in Lower Manhattan ...,New York,"['Verizon Communications', 'Office Buildings a...",2008-01-15 05:00:00+00:00,https://www.nytimes.com/2008/01/15/nyregion/15...,99740,38,171,209,31


In [3]:
# Load reviewed sample
reviewed_sample_path = 'data/samples_reviewed/rule_classification_validation_sample_reviewed.csv'

In [4]:
# --- Model Configuration ---
MODEL_NAME = 'BAAI/bge-m3' # Performed well in the previous phase

# --- k-NN & Classification Parameters ---
K_KEYWORDS = 7 # 7 categories (Finance removed, integrated into Governance and Legal)
# Dynamic k for sentences to adapt to varying article lengths
MIN_K_SENTENCES = 2 # Originally 2 (1st quartal), but 2 is subject to outliers, so set to 5 (which was quite good before when used as a fixed value)
MAX_K_SENTENCES = 26 # As 75% of the articles have 26 sentences or less
PERCENTAGE_K_SENTENCES = 0.2 # 80/20 rule, 20% of the sentences would include the most important information for category classification

# Ambiguity threshold for multi-labeling
CONFIDENCE_THRESHOLD = 0.40 # 0.45 was too strict, leading to many 'Low Confidence' labels
# A review of Low Confidence articles also showed that articles with 0.39 confidence were still relevant, but not all, so 0.4 is a better threshold (old news, since we used only the expanded dictionary)
# Raised from 0.40 to 0.41 for v6, because 0.4 was too lenient (see confmatrix or classification report of v5 in analyze_21.ipynb)
AMBIGUITY_DELTA = 0.05

## Data Loading

Load sentence-level dataset (already preprocessed and cleaned).

In [ ]:
# Load sentence-level dataset
print("=" * 80)
print("PROVISIONAL MULTI-LABELING (v4)")
print("=" * 80)
print(f"\n📊 Configuration:")
print(f"   Dataset: {TRAIN_DATA_PATH}")
print(f"   Dictionary: {KEYWORD_DICTIONARY_PATH}")
print(f"   Categories: {K_KEYWORDS}")
print(f"   Model: {MODEL_NAME}")
print(f"\n⚠️  NOTE: If you see caching errors, delete old embedding files:")
print(f"   - {SENTENCE_EMBEDDINGS_PATH}")
print(f"   - {KEYWORD_EMBEDDINGS_PATH}\n")

sentence_df = pd.read_parquet(TRAIN_DATA_PATH)
print(f"✓ Loaded {len(sentence_df):,} sentences from {sentence_df['article_id'].nunique():,} articles")
print(f"   Avg sentences per article: {len(sentence_df) / sentence_df['article_id'].nunique():.2f}\n")

PROVISIONAL MULTI-LABELING (v4)

📊 Configuration:
   Dataset: data/01_preprocessed/articles_entity_linked_cleaned.parquet
   Dictionary: results/phase i/13_final_dictionary_v4.csv
   Categories: 7
   Model: BAAI/bge-m3

⚠️  NOTE: If you see caching errors, delete old embedding files:
   - embeddings/sentence_embeddings_v4.pt
   - embeddings/keyword_embeddings_v4.pt

✓ Loaded 19659 articles
Creating sentence-level dataset from 19659 articles...


Splitting sentences: 100%|██████████| 19659/19659 [00:01<00:00, 13069.11it/s]


Created 180840 sentences from 19659 articles
Avg sentences per article: 9.20
✓ Created sentence dataset with 180840 sentences



## Helper Functions

In [6]:
def check_file_exists(filepath):
    """Checks if a file exists and exits if it doesn't."""
    if not os.path.exists(filepath):
        print(f"Error: Required file not found at '{filepath}'")
        sys.exit(1)


def load_data():
    """
    Loads keyword dictionaries.
    
    Note: In v4, we preprocess article data to sentence-level above.
    This function now only loads the keyword dictionary.
    """
    print("Loading keyword dictionary...")
    
    check_file_exists(KEYWORD_DICTIONARY_PATH)
    
    keyword_df = pd.read_csv(KEYWORD_DICTIONARY_PATH)
    
    if 'keyword' not in keyword_df.columns:
        print(f"Error: 'keyword' column not found in '{KEYWORD_DICTIONARY_PATH}'")
        sys.exit(1)
    
    keyword_dictionaries = keyword_df.groupby('category')['keyword'].apply(list).to_dict()
    
    print(f"✓ Loaded {len(keyword_df)} keywords across {len(keyword_dictionaries)} categories")
    for cat, kws in sorted(keyword_dictionaries.items()):
        print(f"   {cat}: {len(kws)} keywords")
    
    return keyword_dictionaries


def get_or_create_embeddings(sentences_df, keyword_dictionaries, model):
    """Loads embeddings from cache if they exist, otherwise creates and saves them."""
    print("Checking for cached embeddings...")

    if os.path.exists(SENTENCE_EMBEDDINGS_PATH):
        print(f"Loading cached sentence embeddings from '{SENTENCE_EMBEDDINGS_PATH}'...")
        sentence_embeddings = torch.load(SENTENCE_EMBEDDINGS_PATH)
    else:
        print("No cached sentence embeddings found. Creating new ones...")
        all_sentences = sentences_df['text'].astype(str).tolist()
        sentence_embeddings = model.encode(all_sentences, convert_to_tensor=True, show_progress_bar=True)
        print(f"Saving sentence embeddings to '{SENTENCE_EMBEDDINGS_PATH}'...")
        torch.save(sentence_embeddings, SENTENCE_EMBEDDINGS_PATH)

    if os.path.exists(KEYWORD_EMBEDDINGS_PATH):
        print(f"Loading cached keyword embeddings from '{KEYWORD_EMBEDDINGS_PATH}'...")
        keyword_embeddings_dict = torch.load(KEYWORD_EMBEDDINGS_PATH)
    else:
        print("No cached keyword embeddings found. Creating new ones...")
        keyword_embeddings_dict = {
            cat: model.encode(kws, convert_to_tensor=True)
            for cat, kws in keyword_dictionaries.items()
        }
        print(f"Saving keyword embeddings to '{KEYWORD_EMBEDDINGS_PATH}'...")
        torch.save(keyword_embeddings_dict, KEYWORD_EMBEDDINGS_PATH)
        
    print("Embeddings are ready.")
    return sentence_embeddings, keyword_embeddings_dict


def calculate_sentence_scores(sentence_embeddings, keyword_embeddings_dict, keyword_dictionaries, k):
    """
    Calculates scores and identifies top keywords for each sentence for each category.
    """
    print(f"Calculating sentence scores and keywords with k={k}...")
    
    sentence_embeddings = util.normalize_embeddings(sentence_embeddings)
    sentence_scores = {}
    sentence_keywords = {}
    
    for category, keyword_embeds in tqdm(keyword_embeddings_dict.items(), desc="Scoring Categories"):
        keyword_embeds = util.normalize_embeddings(keyword_embeds)
        category_keywords = keyword_dictionaries[category]
        
        search_results = util.semantic_search(
            sentence_embeddings, keyword_embeds, 
            top_k=min(k, len(keyword_embeds)), 
            score_function=util.dot_score
        )
        
        category_scores = []
        category_top_keywords = []
        
        for hits in search_results:
            # Filter out hits with non-positive scores
            positive_hits = [hit for hit in hits if hit['score'] > 0]
            
            if not positive_hits:
                category_scores.append(0.0)
                category_top_keywords.append([])
                continue
            
            # Calculate average score
            avg_score = sum(hit['score'] for hit in positive_hits) / len(positive_hits)
            category_scores.append(avg_score)
            
            # Get the corresponding keywords
            top_kws = [category_keywords[hit['corpus_id']] for hit in positive_hits]
            category_top_keywords.append(top_kws)
            
        sentence_scores[category] = np.array(category_scores)
        sentence_keywords[category] = category_top_keywords
        
    return pd.DataFrame(sentence_scores), pd.DataFrame(sentence_keywords)


def aggregate_to_article_level(sentences_df, sentence_scores_df, sentence_keywords_df, min_k, max_k, percentage_k):
    """
    Aggregates sentence scores to the article level.
    """
    print(f"Aggregating scores to article level with dynamic k (min={min_k}, max={max_k}, pct={percentage_k})...")
    
    # Combine all data into a single DataFrame for easier processing
    full_df = pd.concat([
        sentences_df.reset_index(drop=True), 
        sentence_scores_df.reset_index(drop=True),
        sentence_keywords_df.add_suffix('_keywords').reset_index(drop=True)
    ], axis=1)
    
    article_scores = {}
    
    grouped = full_df.groupby('article_id')
    
    for article_id, group in tqdm(grouped, desc="Aggregating Articles"):
        num_sentences = len(group)
        k = min(max_k, max(min_k, int(num_sentences * percentage_k)))
        
        scores = {}
        for cat in sentence_scores_df.columns:
            # Get the top k sentences for the category
            top_k_sentences = group.nlargest(k, cat)
            # Calculate the mean score for the top k sentences
            scores[cat] = top_k_sentences[cat].mean()
            
        article_scores[article_id] = scores

    # Create DataFrame from the collected scores
    article_scores_df = pd.DataFrame.from_dict(article_scores, orient='index').fillna(0)
    
    # Return both the aggregated scores and the full sentence-level data for later use
    return article_scores_df, full_df


def apply_classification_rules(scores_df, threshold, delta):
    """Applies the initial multi-labeling and ambiguity rules."""
    print("Applying initial classification rules...")
    provisional_labels = []
    provisional_categories_list = []
    
    for _, row in tqdm(scores_df.iterrows(), total=len(scores_df), desc="Classifying Articles"):
        high_scores = row[row >= threshold].sort_values(ascending=False)

        if len(high_scores) == 0:
            provisional_labels.append("Low Confidence")
            provisional_categories_list.append(None)
        elif len(high_scores) == 1:
            provisional_labels.append("Single Label")
            provisional_categories_list.append(high_scores.index[0])
        else:
            if (high_scores.iloc[0] - high_scores.iloc[1]) < delta:
                provisional_labels.append("Complex Event (High Confidence, Low Delta)")
                top_score = high_scores.iloc[0]
                close_categories = high_scores[high_scores >= top_score - delta]
                provisional_categories_list.append(", ".join(close_categories.index))
            else:
                provisional_labels.append("Complex Event (High Confidence, High Delta)")
                provisional_categories_list.append(", ".join(high_scores.index))
                
    scores_df['provisional_label'] = provisional_labels
    scores_df['provisional_categories'] = provisional_categories_list
    return scores_df


def get_final_category(df):
    """
    Determines the final category based on a tiered priority system,
    defaulting to the highest score if ambiguity is only within Tier 3.
    
    Updated for v4: 7 categories (Finance removed).
    """
    print("Applying tiered priority system for final category selection...")
    
    # Updated category names for v4 (short names, 7 categories)
    tier_1 = ['Governance', 'Personnel']
    tier_2 = ['Products', 'IT/Data', 'Processes']
    tier_3 = ['Legal', 'Communication']
    ordered_categories = tier_1 + tier_2 + tier_3

    def find_final_category(row):
        label = row['provisional_label']
        categories_str = row['provisional_categories']

        if pd.isna(categories_str) or label == 'Low Confidence':
            return 'No Event'
        
        potential_categories = {cat.strip() for cat in categories_str.split(',')}

        # For these labels, the highest score is the clear winner.
        if label in ['Single Label', 'Complex Event (High Confidence, High Delta)']:
            # Ensure there are categories before trying to access them
            if potential_categories:
                return categories_str.split(',')[0].strip()
            else:
                return 'No Event'
        
        if label == 'Complex Event (High Confidence, Low Delta)':
            # Check for Tier 1 categories first
            for category in ordered_categories:
                if category in tier_1 and category in potential_categories:
                    return category # Return the highest-priority Tier 1 category found
            
            # If no Tier 1, check for Tier 2
            for category in ordered_categories:
                if category in tier_2 and category in potential_categories:
                    return category # Return the highest-priority Tier 2 category found

            # If ambiguity is only within Tier 3, pick the one with the highest score
            return categories_str.split(',')[0].strip()
        
        return None # Fallback for any unhandled cases

    df['final_category'] = df.apply(find_final_category, axis=1)
    return df

def extract_final_details(classified_df, full_sentence_df, min_k, max_k, percentage_k):
    """
    Extracts top sentences and keywords for only the final_category of each article.
    """
    print("Extracting final details for the chosen category...")
    
    # Prepare for collecting details
    details_list = []
    
    # Group the sentence data once
    grouped_sentences = full_sentence_df.groupby('article_id')
    
    # Iterate through the classified articles
    for article_id, row in tqdm(classified_df.iterrows(), total=len(classified_df), desc="Extracting Details"):
        final_cat = row['final_category']
        
        if pd.isna(final_cat) or final_cat == 'No Event':
            details_list.append({'most_relevant_sentences': None, 'most_relevant_keywords': None})
            continue
            
        # Get the corresponding group of sentences
        group = grouped_sentences.get_group(article_id)
        
        # Determine k for this article
        num_sentences = len(group)
        k = min(max_k, max(min_k, int(num_sentences * percentage_k)))
        
        # Get the top k sentences for the final category
        top_k_sentences = group.nlargest(k, final_cat)
        
        # Extract sentences and keywords
        sentences = " | ".join(top_k_sentences['text'].tolist())
        keywords_list = top_k_sentences[f'{final_cat}_keywords'].sum()
        keywords = ", ".join(sorted(list(set(keywords_list))))
        
        details_list.append({'most_relevant_sentences': sentences, 'most_relevant_keywords': keywords})

    # Create a DataFrame from the list of details
    details_df = pd.DataFrame(details_list, index=classified_df.index)
    
    # Merge the details back into the classified DataFrame
    return classified_df.join(details_df)

def main(output_path, sentences_df):
    """Main function to run the entire provisional labeling pipeline."""
    
    # Load keyword dictionary
    keyword_dictionaries = load_data()
    
    # Load model and set device to GPU if available
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    print(f"Loading sentence-transformer model: {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME, device=device)
    
    # Get or create embeddings
    sentence_embeds, keyword_embeds_dict = get_or_create_embeddings(sentences_df, keyword_dictionaries, model)
    
    # Phase 1: Sentence-level scoring and keyword identification
    sentence_scores_df, sentence_keywords_df = calculate_sentence_scores(
        sentence_embeds, keyword_embeds_dict, keyword_dictionaries, K_KEYWORDS
    )
    
    # Phase 2: Article-level aggregation (now returns scores and full sentence data)
    article_scores_df, full_sentence_df = aggregate_to_article_level(
        sentences_df, 
        sentence_scores_df, 
        sentence_keywords_df,
        MIN_K_SENTENCES, 
        MAX_K_SENTENCES, 
        PERCENTAGE_K_SENTENCES
    )
    
    # Stage 1: Initial Classification
    classified_df = apply_classification_rules(article_scores_df, CONFIDENCE_THRESHOLD, AMBIGUITY_DELTA)
    
    # Stage 2: Tiered Resolution (now only adds 'final_category' column)
    classified_df = get_final_category(classified_df)

    # Stage 3: Efficiently extract details for the final category ONLY
    final_df = extract_final_details(
        classified_df, 
        full_sentence_df,
        MIN_K_SENTENCES, 
        MAX_K_SENTENCES, 
        PERCENTAGE_K_SENTENCES
    )
    
    # Combine with original text for context
    article_text_df = sentences_df.groupby('article_id')['text'].apply(' '.join).reset_index()
    result_df = article_text_df.merge(final_df, left_on='article_id', right_index=True)
    
    # Save the final labeled dataset to a Parquet file
    result_df.to_parquet(output_path, index=False)
    print(f"✓ Results saved to '{output_path}'")
    print("\n--- Provisional Labeling Complete ---")

In [6]:
# Analysis function 

def analyze_classification_results(reviewed_sample_path, classified_df_path):
    # Load the reviewed sample
    reviewed_sample = pd.read_csv(reviewed_sample_path)
    reviewed_sample = reviewed_sample[['article_id', 'text', 'actual_label']]

    # Load the classified DataFrame
    classified_df = pd.read_parquet(classified_df_path)
    classified_df = classified_df[['article_id', 'final_category', 'provisional_label', 'provisional_categories']]

    # Merge the reviewed sample with the classified DataFrame
    merged_df = pd.merge(reviewed_sample, classified_df, on='article_id')

    # Fill in NaN values in 'final_category' with 'No Event'
    merged_df['final_category'].fillna('No Event', inplace=True)

    # Classification report
    y_true = merged_df['actual_label']
    y_pred = merged_df['final_category']

    print("Classification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    # Visualize the confusion matrix
    import matplotlib.pyplot as plt

    print("\nConfusion Matrix:")
    
    # Get a sorted list of unique labels from both true and predicted values
    labels = np.union1d(y_true.unique(), y_pred.unique())
    
    # Calculate confusion matrix with the sorted labels
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    # Plotting the confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title('Confusion Matrix')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Draw ROC curve
    # Binarize the labels for ROC curve
    y_true_binarized = label_binarize(y_true, classes=labels)
    y_pred_binarized = label_binarize(y_pred, classes=labels)
    n_classes = y_true_binarized.shape[1]
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_binarized[:, i], y_pred_binarized[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    # Plot ROC curve
    plt.figure(figsize=(10, 8))
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f'ROC curve of class {labels[i]} (area = {roc_auc[i]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--',
                label='Random guess (area = 0.50)', color='gray')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc='lower right')
    plt.grid()
    plt.show()

In [8]:
# Run provisionalclassification (v4)
if __name__ == '__main__':
    print("\n" + "=" * 80)
    print("RUNNING PROVISIONAL MULTI-LABELING (v4)")
    print("=" * 80)
    main(OUTPUT_PATH, sentence_df)


RUNNING PROVISIONAL MULTI-LABELING (v4)
Loading keyword dictionary...
✓ Loaded 1109 keywords across 7 categories
   Communication: 172 keywords
   Governance: 202 keywords
   IT/Data: 151 keywords
   Legal: 183 keywords
   Personnel: 180 keywords
   Processes: 100 keywords
   Products: 121 keywords
Using device: cuda
Loading sentence-transformer model: BAAI/bge-m3...
Checking for cached embeddings...
Loading cached sentence embeddings from 'embeddings/sentence_embeddings_v4.pt'...
Loading cached keyword embeddings from 'embeddings/keyword_embeddings_v4.pt'...
Embeddings are ready.
Calculating sentence scores and keywords with k=7...


Scoring Categories: 100%|██████████| 7/7 [00:16<00:00,  2.35s/it]


Aggregating scores to article level with dynamic k (min=2, max=26, pct=0.2)...


Aggregating Articles: 100%|██████████| 19659/19659 [02:04<00:00, 158.29it/s]


Applying initial classification rules...


Classifying Articles: 100%|██████████| 19659/19659 [00:05<00:00, 3387.61it/s]


Applying tiered priority system for final category selection...
Extracting final details for the chosen category...


Extracting Details: 100%|██████████| 19659/19659 [00:21<00:00, 927.15it/s]


✓ Results saved to 'results/phase ii/21_provisionally_labeled_train_set_v4.parquet'

--- Provisional Labeling Complete ---


## Summary of v4 Changes

**Updates for v4:**
1. ✅ Dataset: Using sentence-level cleaned entity-linked data (163,139 sentences from 19,659 articles)
2. ✅ Dictionary: Using final v4 dictionary (1,109 keywords, 7 categories)
3. ✅ Categories: Reduced from 8 → 7 (Finance removed, integrated into Governance & Legal)
4. ✅ Data loading: Direct loading of preprocessed sentence-level data
5. ✅ Embedding cache: Using v4 paths to match updated dataset
6. ✅ Tiered priority system: Updated with new short category names

**Category Changes:**
- OLD: Strategy and governance → NEW: Governance
- OLD: Products and services → NEW: Products  
- OLD: Information technology and data management → NEW: IT/Data
- OLD: Processes and supply chains → NEW: Processes
- OLD: Legality and regulation → NEW: Legal
- OLD: Financial performance → **REMOVED** (integrated into Governance & Legal)
- OLD: Communication and media → NEW: Communication

**Tiered Priority System (v4):**
- **Tier 1** (highest priority): Governance, Personnel
- **Tier 2**: Products, IT/Data, Processes
- **Tier 3**: Legal, Communication